In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

print("All libraries imported!")

All libraries imported!


In [7]:
df = pd.read_csv('languages.csv')
print("shape:",df.shape)
print("\ncolumns:",df.columns.tolist())

shape: (2722, 15)

columns: ['ID', 'Name in English', 'Name in French', 'Name in Spanish', 'Countries', 'Country codes alpha 3', 'ISO639-3 codes', 'Degree of endangerment', 'Alternate names', 'Name in the language', 'Number of speakers', 'Sources', 'Latitude', 'Longitude', 'Description of the location']


In [8]:
df.head()

,ID,Name in English,Name in French,Name in Spanish,Countries,Country codes alpha 3,ISO639-3 codes,Degree of endangerment,Alternate names,Name in the language,Number of speakers,Sources,Latitude,Longitude,Description of the location
0,1022,South Italian,italien du sud,napolitano-calabrés,Italy,ITA,nap,Vulnerable,Neapolitan; Neapolitan-Calabrese; неаполитанск...,NaN,7500000.0,NaN,40.9798,15.2490,"Campania, Lucania (Basilicata), Abruzzi (Abruz..."
1,1023,Sicilian,sicilien,siciliano,Italy,ITA,scn,Vulnerable,NaN,NaN,5000000.0,NaN,37.4399,14.5019,"Sicily (Sicilia), southern and central Calabri..."
2,383,Low Saxon,bas-saxon,bajo sajón,"Germany, Denmark, Netherlands, Poland, Russian...","DEU, DNK, NLD, POL, RUS","act, drt, frs, gos, nds, sdz, stl, twd, vel, wep",Vulnerable,"Low German, Niedersächsisch, Nedersaksisch, Ni...",Neddersassisch,4800000.0,NaN,53.4029,10.3601,"northern Germany, the north-eastern part of th..."
3,335,Belarusian,biélorusse,bielorruso,"Belarus, Latvia, Lithuania, Poland, Russian Fe...","BRB, LVA, LTU, POL, RUS, UKR",bel,Vulnerable,NaN,NaN,4000000.0,Hienadź Cychun: Weißrussisch. — Lexikon der Sp...,53.9560,27.5756,Belarus except the Polesian-speaking south-wes...
4,382,Lombard,lombard,lombardo,"Italy, Switzerland","ITA, CHE",lmo,Definitely endangered,NaN,NaN,3500000.0,NaN,45.7215,9.3273,the region of Lombardy (except the southernmos...


In [9]:
df = df[['Name in English', 'Number of speakers', 'Latitude', 'Longitude', 'Degree of endangerment']]
print("New shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

New shape: (2722, 5)

First 5 rows:
  Name in English  Number of speakers  Latitude  Longitude  \
0   South Italian           7500000.0   40.9798    15.2490   
1        Sicilian           5000000.0   37.4399    14.5019   
2       Low Saxon           4800000.0   53.4029    10.3601   
3      Belarusian           4000000.0   53.9560    27.5756   
4         Lombard           3500000.0   45.7215     9.3273   

  Degree of endangerment  
0             Vulnerable  
1             Vulnerable  
2             Vulnerable  
3             Vulnerable  
4  Definitely endangered  


In [11]:
print("Missing values in each column:")
print(df.isnull().sum())

Missing values in each column:
Name in English             0
Number of speakers        183
Latitude                    3
Longitude                   3
Degree of endangerment      0
dtype: int64


In [12]:
df['Number of speakers'] = df['Number of speakers'].fillna(df['Number of speakers'].median())
df['Latitude'] = df['Latitude'].fillna(df['Latitude'].mean())
df['Longitude'] = df['Longitude'].fillna(df['Longitude'].mean())
print("Missing values after fixing:")
print(df.isnull().sum())

Missing values after fixing:
Name in English           0
Number of speakers        0
Latitude                  0
Longitude                 0
Degree of endangerment    0
dtype: int64


In [13]:
# List of labels that mean "Endangered"
endangered_labels = ['Critically endangered', 'Severely endangered', 'Definitely endangered']

# Create new column: 1 if endangered, 0 if not
df['target'] = df['Degree of endangerment'].apply(lambda x: 1 if x in endangered_labels else 0)

print("Target distribution:")
print(df['target'].value_counts())
print("\nWhat 0 and 1 mean:")
print(df[['Degree of endangerment', 'target']].drop_duplicates().sort_values('target'))

Target distribution:
target
1    1841
0     881
Name: count, dtype: int64

What 0 and 1 mean:
    Degree of endangerment  target
0               Vulnerable       0
485                Extinct       0
4    Definitely endangered       1
48     Severely endangered       1
232  Critically endangered       1


In [14]:
# X = features (what the model uses to predict)
X = df[['Number of speakers', 'Latitude', 'Longitude']]

# y = target (what we want to predict)
y = df['target']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFirst 3 rows of X:")
print(X.head(3))
print("\nFirst 3 values of y:")
print(y.head(3))

X shape: (2722, 3)
y shape: (2722,)

First 3 rows of X:
   Number of speakers  Latitude  Longitude
0           7500000.0   40.9798    15.2490
1           5000000.0   37.4399    14.5019
2           4800000.0   53.4029    10.3601

First 3 values of y:
0    0
1    0
2    0
Name: target, dtype: int64


In [15]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Before scaling (first row):")
print(X.iloc[0].values)

print("\nAfter scaling (first row):")
print(X_scaled[0])

Before scaling (first row):
[7.50000e+06 4.09798e+01 1.52490e+01]

After scaling (first row):
[24.74702793  1.0648964  -0.07582416]


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("Training set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)

Training set: (2177, 3) (2177,)
Test set: (545, 3) (545,)


In [17]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Model trained!")

Model trained!


In [18]:
y_pred = model.predict(X_test)

print("Predictions made!")
print("First 10 predictions:", y_pred[:10])
print("First 10 actual answers:", y_test.values[:10])

Predictions made!
First 10 predictions: [1 1 0 1 1 1 1 1 1 0]
First 10 actual answers: [1 0 0 0 1 1 1 1 1 0]


In [19]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Endangered', 'Endangered']))

Accuracy: 0.8330275229357799

Confusion Matrix:
[[117  50]
 [ 41 337]]

Classification Report:
                precision    recall  f1-score   support

Not Endangered       0.74      0.70      0.72       167
    Endangered       0.87      0.89      0.88       378

      accuracy                           0.83       545
     macro avg       0.81      0.80      0.80       545
  weighted avg       0.83      0.83      0.83       545



In [20]:
# Look up "Koro" in the data
koro = df[df['Name in English'] == 'Koro'].iloc[0]

print("Language:", koro['Name in English'])
print("Speakers:", koro['Number of speakers'])
print("Location:", koro['Latitude'], koro['Longitude'])
print("Actual status:", koro['Degree of endangerment'])

# Prepare for prediction
koro_features = np.array([[koro['Number of speakers'], koro['Latitude'], koro['Longitude']]])
koro_scaled = scaler.transform(koro_features)
prediction = model.predict(koro_scaled)[0]

print("\nModel prediction:", "Endangered" if prediction == 1 else "Not Endangered")

Language: Koro
Speakers: 160.0
Location: -14.3242 167.4728
Actual status: Definitely endangered

Model prediction: Endangered


C:\Users\manoj\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
